In [1]:
# ============================================================
# ANALÍTICA PUMA — EVALUACIÓN INDIVIDUAL DE JUGADORES

# ============================================================

from pathlib import Path

import duckdb
import numpy as np
import pandas as pd

# ------------------------------------------------------------
# CONFIGURACIÓN DEL PROYECTO
# ------------------------------------------------------------

DB_PATH = Path("../data/analitica_puma.duckdb")

# ------------------------------------------------------------
# VERIFICACIÓN DEL ENTORNO
# ------------------------------------------------------------

print("=== ENTORNO ===")
print("Pandas:", pd.__version__)
print("NumPy:", np.__version__)
print("DuckDB:", duckdb.__version__)

print("\n=== BASE DE DATOS ===")
print("Ruta:", DB_PATH)
print("Existe:", DB_PATH.exists())

assert DB_PATH.exists(), (
    f"No se encontró la base de datos en: {DB_PATH.resolve()}"
)

# Conexión exclusivamente de lectura
con = duckdb.connect(
    str(DB_PATH),
    read_only=True,
)

tables = con.execute("SHOW TABLES").df()

print("\n=== TABLAS DISPONIBLES ===")
print(tables.to_string(index=False))

print("\n✓ Entorno preparado correctamente.")

=== ENTORNO ===
Pandas: 3.0.5
NumPy: 2.5.3
DuckDB: 1.5.5

=== BASE DE DATOS ===
Ruta: ../data/analitica_puma.duckdb
Existe: True

=== TABLAS DISPONIBLES ===
              name
  match_team_stats
           matches
player_match_stats
           players
           seasons
             teams

✓ Entorno preparado correctamente.


In [2]:
# ============================================================
# PASO 2 — CARGA DE DATOS RAW PARA EL ANÁLISIS
# ============================================================

df = con.execute("""
    SELECT
        pms.*,
        p.name AS player_name,
        p.position_id,
        p.detailed_position_id
    FROM player_match_stats AS pms
    JOIN players AS p
        ON pms.player_id = p.id
    ORDER BY
        pms.match_id,
        pms.team_id,
        pms.player_id
""").df()

print("=== DATASET RAW ===")
print("Shape:", df.shape)

print("Partidos:", df["match_id"].nunique())
print("Jugadores:", df["player_id"].nunique())
print("Equipos:", df["team_id"].nunique())

print("\nMinutos:")
print(df["minutes"].describe())

print("\nDuplicados player × match:")
print(
    df.duplicated(
        subset=["match_id", "player_id"]
    ).sum()
)

print("\n✓ Datos raw cargados desde DuckDB.")

=== DATASET RAW ===
Shape: (1023, 49)
Partidos: 34
Jugadores: 393
Equipos: 18

Minutos:
count    1023.000000
mean       65.523949
std        30.575548
min         1.000000
25%        41.000000
50%        83.000000
75%        90.000000
max        90.000000
Name: minutes, dtype: float64

Duplicados player × match:
0

✓ Datos raw cargados desde DuckDB.


In [3]:
# ============================================================
# PASO 3 — POSICIÓN GENERAL Y PARTICIPACIONES VÁLIDAS
# ============================================================

# Sportmonks position_id:
# 24 = Goalkeeper
# 25 = Defender
# 26 = Midfielder
# 27 = Attacker

POSITION_MAP = {
    24: "GK",
    25: "DF",
    26: "MF",
    27: "FW",
}

MIN_APPEARANCE_MINUTES = 30


# ------------------------------------------------------------
# 1. POSICIÓN GENERAL
# ------------------------------------------------------------

df["position_group"] = df["position_id"].map(POSITION_MAP)


# ------------------------------------------------------------
# 2. PARTICIPACIONES VÁLIDAS
# ------------------------------------------------------------

df_valid = (
    df.loc[
        df["minutes"] >= MIN_APPEARANCE_MINUTES
    ]
    .copy()
)


# ------------------------------------------------------------
# 3. CONTROLES
# ------------------------------------------------------------

print("=== PARTICIPACIONES VÁLIDAS ===")

print("Umbral:", MIN_APPEARANCE_MINUTES, "minutos")
print("Shape raw:", df.shape)
print("Shape válido:", df_valid.shape)

print(
    "Participaciones excluidas:",
    len(df) - len(df_valid)
)

print(
    "Partidos:",
    df_valid["match_id"].nunique()
)

print(
    "Jugadores:",
    df_valid["player_id"].nunique()
)

print("\n=== POSICIONES ===")

print(
    df_valid["position_group"]
    .value_counts(dropna=False)
    .sort_index()
)

print(
    "\nPosición general NULL:",
    df_valid["position_group"].isna().sum()
)

print(
    "\n¿Base válida reconstruida correctamente?:",
    len(df_valid) == 808
    and df_valid["position_group"].isna().sum() == 0
)

=== PARTICIPACIONES VÁLIDAS ===
Umbral: 30 minutos
Shape raw: (1023, 50)
Shape válido: (808, 50)
Participaciones excluidas: 215
Partidos: 34
Jugadores: 322

=== POSICIONES ===
position_group
DF    304
FW    219
GK     67
MF    218
Name: count, dtype: int64

Posición general NULL: 0

¿Base válida reconstruida correctamente?: True


In [4]:
# ============================================================
# PASO 4 — LIMPIEZA DEFINITIVA DE VARIABLES
# ============================================================

# ------------------------------------------------------------
# 1. EXCLUIR REGISTRO INCONSISTENTE
# ------------------------------------------------------------
# Diego Campillo:
# dribble_attempts = NULL
# successful_dribbles = 1
#
# No podemos reconstruir de forma fiable el número de intentos,
# por lo que excluimos esta participación completa.

mask_campillo = (
    (df_valid["match_id"] == 30)
    & (df_valid["player_id"] == 318)
    & (df_valid["team_id"] == 3)
)

campillo_excluded = df_valid.loc[
    mask_campillo,
    [
        "match_id",
        "player_id",
        "player_name",
        "team_id",
        "minutes",
        "dribble_attempts",
        "successful_dribbles",
    ],
].copy()

df_valid = df_valid.loc[~mask_campillo].copy()


# ------------------------------------------------------------
# 2. VARIABLES DONDE NULL REPRESENTA AUSENCIA DE EVENTO
# ------------------------------------------------------------

ZERO_EVENT_COLS = [
    "goals",
    "assists",
    "shots",
    "shots_on_target",
    "key_passes",
    "total_crosses",
    "accurate_crosses",
    "dribble_attempts",
    "successful_dribbles",
    "passes_in_final_third",
    "total_duels",
    "duels_won",
    "duels_lost",
    "long_balls",
    "long_balls_won",
    "tackles",
    "interceptions",
    "ball_recoveries",
    "clearances",
    "dribbled_past",
    "fouls",
    "fouls_drawn",
    "yellow_cards",
    "red_cards",
    "backward_passes",
    "possession_lost",
]

for col in ZERO_EVENT_COLS:
    df_valid[f"{col}_adj"] = df_valid[col].fillna(0)


# ------------------------------------------------------------
# 3. PASES Y TOQUES
# ------------------------------------------------------------


for col in [
    "passes",
    "accurate_passes",
    "touches",
]:
    df_valid[f"{col}_adj"] = df_valid[col]


# ------------------------------------------------------------
# 4. DISPOSSESSED
# ------------------------------------------------------------
# Para jugadores de campo:
# NULL se interpreta como ausencia del evento.
#
# Para porteros:
# permanece NA por considerarse estructural/no aplicable.

df_valid["dispossessed_adj"] = df_valid["dispossessed"]

field_player = df_valid["position_group"] != "GK"

df_valid.loc[
    field_player & df_valid["dispossessed_adj"].isna(),
    "dispossessed_adj",
] = 0


# ------------------------------------------------------------
# 5. DUELOS AÉREOS
# ------------------------------------------------------------
# Tratamiento conservador:
#
# - won presente + lost NULL  -> lost = 0
# - lost presente + won NULL  -> won = 0
# - ambos NULL                -> permanecen NA
#
# No fabricamos actividad aérea cuando Sportmonks no reportó
# ninguno de los componentes.

df_valid["aerials_won_adj"] = df_valid["aerials_won"]
df_valid["aerials_lost_adj"] = df_valid["aerials_lost"]

mask = (
    df_valid["aerials_won_adj"].notna()
    & df_valid["aerials_lost_adj"].isna()
)
df_valid.loc[mask, "aerials_lost_adj"] = 0

mask = (
    df_valid["aerials_lost_adj"].notna()
    & df_valid["aerials_won_adj"].isna()
)
df_valid.loc[mask, "aerials_won_adj"] = 0

df_valid["aerials_adj"] = (
    df_valid["aerials_won_adj"]
    + df_valid["aerials_lost_adj"]
)


# ------------------------------------------------------------
# 6. MÉTRICAS EXCLUSIVAS DE PORTEROS
# ------------------------------------------------------------
# Para GK:
# NULL en saves / saves_inside_box se interpreta como 0.
# NULL en goals_conceded fue auditado como clean sheet -> 0.
#
# Para jugadores de campo:
# estas métricas permanecen NA porque no son aplicables.

GK_COLS = [
    "saves",
    "saves_inside_box",
    "goals_conceded",
]

is_gk = df_valid["position_group"] == "GK"

for col in GK_COLS:
    df_valid[f"{col}_adj"] = np.nan

    df_valid.loc[
        is_gk,
        f"{col}_adj",
    ] = df_valid.loc[
        is_gk,
        col,
    ].fillna(0)


# ------------------------------------------------------------
# 7. VARIABLES AJUSTADAS DEFINITIVAS
# ------------------------------------------------------------

ADJUSTED_COLS = [
    "accurate_crosses_adj",
    "accurate_passes_adj",
    "aerials_adj",
    "aerials_lost_adj",
    "aerials_won_adj",
    "assists_adj",
    "backward_passes_adj",
    "ball_recoveries_adj",
    "clearances_adj",
    "dispossessed_adj",
    "dribble_attempts_adj",
    "dribbled_past_adj",
    "duels_lost_adj",
    "duels_won_adj",
    "fouls_adj",
    "fouls_drawn_adj",
    "goals_adj",
    "goals_conceded_adj",
    "interceptions_adj",
    "key_passes_adj",
    "long_balls_adj",
    "long_balls_won_adj",
    "passes_adj",
    "passes_in_final_third_adj",
    "possession_lost_adj",
    "red_cards_adj",
    "saves_adj",
    "saves_inside_box_adj",
    "shots_adj",
    "shots_on_target_adj",
    "successful_dribbles_adj",
    "tackles_adj",
    "total_crosses_adj",
    "total_duels_adj",
    "touches_adj",
    "yellow_cards_adj",
]


# ------------------------------------------------------------
# 8. CONTROLES DE REPRODUCIBILIDAD
# ------------------------------------------------------------

missing_adjusted = [
    col
    for col in ADJUSTED_COLS
    if col not in df_valid.columns
]

null_adjusted = (
    df_valid[ADJUSTED_COLS]
    .isna()
    .sum()
    .sort_values(ascending=False)
)

print("=== REGISTRO EXCLUIDO ===")
print(campillo_excluded.to_string(index=False))

print("\n=== CONTROL DE LIMPIEZA ===")
print("Shape df_valid:", df_valid.shape)
print("Variables ajustadas:", len(ADJUSTED_COLS))
print("Variables faltantes:", missing_adjusted)

print("\n=== NULL EN VARIABLES AJUSTADAS ===")
print(null_adjusted[null_adjusted > 0])

cleaning_ok = (
    len(df_valid) == 807
    and len(ADJUSTED_COLS) == 36
    and len(missing_adjusted) == 0
    and len(campillo_excluded) == 1
)

print(
    "\n¿Limpieza reconstruida correctamente?:",
    cleaning_ok,
)

assert cleaning_ok, (
    "La limpieza no reproduce el estado esperado."
)

=== REGISTRO EXCLUIDO ===
 match_id  player_id              player_name  team_id  minutes  dribble_attempts  successful_dribbles
       30        318 Diego Campillo del Campo        3       90              <NA>                    1

=== CONTROL DE LIMPIEZA ===
Shape df_valid: (807, 86)
Variables ajustadas: 36
Variables faltantes: []

=== NULL EN VARIABLES AJUSTADAS ===
saves_adj               740
saves_inside_box_adj    740
goals_conceded_adj      740
aerials_lost_adj        233
aerials_adj             233
aerials_won_adj         233
dispossessed_adj         67
dtype: int64

¿Limpieza reconstruida correctamente?: True


In [5]:
# ============================================================
# PASO 5 — ASIGNACIÓN DE TORNEO
# ============================================================

# ------------------------------------------------------------
# 1. OBTENER TEMPORADA Y TORNEO DESDE MATCHES
# ------------------------------------------------------------

match_tournaments = con.execute("""
    SELECT
        m.id AS match_id,
        m.season_id,
        s.name AS season_name,
        CASE
            WHEN EXTRACT(MONTH FROM m.date) BETWEEN 1 AND 6
                THEN 'Clausura'
            WHEN EXTRACT(MONTH FROM m.date) BETWEEN 7 AND 12
                THEN 'Apertura'
        END AS tournament_name
    FROM matches AS m
    JOIN seasons AS s
        ON m.season_id = s.id
""").df()


# ------------------------------------------------------------
# 2. INCORPORAR TORNEO A LAS PARTICIPACIONES
# ------------------------------------------------------------

df_valid = df_valid.drop(
    columns=[
        "season_id",
        "season_name",
        "tournament_name",
    ],
    errors="ignore",
)

df_valid = df_valid.merge(
    match_tournaments,
    on="match_id",
    how="left",
    validate="many_to_one",
)


# ------------------------------------------------------------
# 3. CONTROLES
# ------------------------------------------------------------

matches_by_tournament = (
    df_valid[
        [
            "match_id",
            "season_name",
            "tournament_name",
        ]
    ]
    .drop_duplicates()
    .groupby(
        ["season_name", "tournament_name"]
    )
    .size()
)

appearances_by_tournament = (
    df_valid
    .groupby(
        ["season_name", "tournament_name"]
    )
    .size()
)

print("=== PARTIDOS POR TORNEO ===")
print(matches_by_tournament)

print("\n=== PARTICIPACIONES VÁLIDAS ===")
print(appearances_by_tournament)

print("\n=== CONTROL ===")
print("Shape df_valid:", df_valid.shape)
print("Partidos únicos:", df_valid["match_id"].nunique())
print(
    "Torneos:",
    sorted(df_valid["tournament_name"].dropna().unique())
)
print(
    "Torneo NULL:",
    df_valid["tournament_name"].isna().sum()
)


tournament_ok = (
    len(df_valid) == 807
    and df_valid["match_id"].nunique() == 34
    and df_valid["tournament_name"].isna().sum() == 0
    and set(df_valid["tournament_name"].unique())
        == {"Apertura", "Clausura"}
)

print(
    "\n¿Asignación correcta?:",
    tournament_ok,
)

assert tournament_ok, (
    "La asignación de torneos no reproduce "
    "el estado esperado."
)

=== PARTIDOS POR TORNEO ===
season_name  tournament_name
2025/2026    Apertura           17
             Clausura           17
dtype: int64

=== PARTICIPACIONES VÁLIDAS ===
season_name  tournament_name
2025/2026    Apertura           397
             Clausura           410
dtype: int64

=== CONTROL ===
Shape df_valid: (807, 89)
Partidos únicos: 34
Torneos: ['Apertura', 'Clausura']
Torneo NULL: 0

¿Asignación correcta?: True


In [6]:
# ============================================================
# PASO 6 — AGREGACIÓN JUGADOR × TORNEO
# ============================================================

# ------------------------------------------------------------
# 1. CLAVES DE LA UNIDAD ANALÍTICA
# ------------------------------------------------------------

GROUP_COLS = [
    "season_id",
    "season_name",
    "tournament_name",
    "player_id",
    "player_name",
    "team_id",
    "position_group",
]


# ------------------------------------------------------------
# 2. APARICIONES Y MINUTOS VÁLIDOS
# ------------------------------------------------------------

player_tournament_features = (
    df_valid
    .groupby(
        GROUP_COLS,
        as_index=False,
    )
    .agg(
        appearances=("match_id", "nunique"),
        total_valid_minutes=("minutes", "sum"),
    )
)


# ------------------------------------------------------------
# 3. ACUMULAR LAS 36 VARIABLES AJUSTADAS
# ------------------------------------------------------------
# min_count=1 es importante:
# si todos los registros de una métrica son NA,
# el total permanece NA en lugar de convertirse en cero.

tournament_metric_totals = (
    df_valid
    .groupby(
        GROUP_COLS,
        as_index=False,
    )[ADJUSTED_COLS]
    .sum(min_count=1)
)


# ------------------------------------------------------------
# 4. RENOMBRAR COMO total_*
# ------------------------------------------------------------

metric_rename_map = {
    col: f"total_{col.removesuffix('_adj')}"
    for col in ADJUSTED_COLS
}

tournament_metric_totals = (
    tournament_metric_totals
    .rename(columns=metric_rename_map)
)

TOTAL_COLS = list(metric_rename_map.values())


# ------------------------------------------------------------
# 5. UNIR TOTALES AL DATASET JUGADOR × TORNEO
# ------------------------------------------------------------

player_tournament_features = (
    player_tournament_features
    .merge(
        tournament_metric_totals,
        on=GROUP_COLS,
        how="left",
        validate="one_to_one",
    )
)


# ------------------------------------------------------------
# 6. CONTROLES DE REPRODUCIBILIDAD
# ------------------------------------------------------------

duplicates = (
    player_tournament_features
    .duplicated(
        subset=GROUP_COLS
    )
    .sum()
)

records_by_tournament = (
    player_tournament_features
    .groupby(
        ["season_name", "tournament_name"]
    )
    .size()
)

minutes_by_tournament = (
    player_tournament_features
    .groupby(
        ["season_name", "tournament_name"]
    )["total_valid_minutes"]
    .sum()
)

print("=== PLAYER × TOURNAMENT ===")
print("Shape:", player_tournament_features.shape)
print("Registros:", len(player_tournament_features))
print("Duplicados:", duplicates)
print("Variables total_*:", len(TOTAL_COLS))

print("\n=== REGISTROS POR TORNEO ===")
print(records_by_tournament)

print("\n=== MINUTOS VÁLIDOS POR TORNEO ===")
print(minutes_by_tournament)

aggregation_ok = (
    len(player_tournament_features) == 448
    and duplicates == 0
    and len(TOTAL_COLS) == 36
    and records_by_tournament.get(
        ("2025/2026", "Apertura"), 0
    ) == 223
    and records_by_tournament.get(
        ("2025/2026", "Clausura"), 0
    ) == 225
)

print(
    "\n¿Agregación correcta?:",
    aggregation_ok,
)

assert aggregation_ok, (
    "La agregación jugador × torneo "
    "no reproduce el estado esperado."
)

=== PLAYER × TOURNAMENT ===
Shape: (448, 45)
Registros: 448
Duplicados: 0
Variables total_*: 36

=== REGISTROS POR TORNEO ===
season_name  tournament_name
2025/2026    Apertura           223
             Clausura           225
dtype: int64

=== MINUTOS VÁLIDOS POR TORNEO ===
season_name  tournament_name
2025/2026    Apertura           31947
             Clausura           31866
Name: total_valid_minutes, dtype: int32

¿Agregación correcta?: True


In [7]:
# ============================================================
# PASO 7 — FEATURES DERIVADAS A NIVEL JUGADOR × TORNEO
# ============================================================

# ------------------------------------------------------------
# 1. MÉTRICAS POR 90 MINUTOS
# ------------------------------------------------------------

PER90_SOURCE_COLS = [
    # Distribución / posesión
    "passes",
    "accurate_passes",
    "key_passes",
    "passes_in_final_third",
    "backward_passes",
    "touches",
    "possession_lost",
    "dispossessed",

    # Ataque
    "goals",
    "assists",
    "shots",
    "shots_on_target",
    "dribble_attempts",
    "successful_dribbles",
    "total_crosses",
    "accurate_crosses",

    # Defensa / duelos
    "tackles",
    "interceptions",
    "ball_recoveries",
    "clearances",
    "dribbled_past",
    "total_duels",
    "duels_won",
    "duels_lost",
    "aerials",
    "aerials_won",
    "aerials_lost",

    # Disciplina
    "fouls",
    "fouls_drawn",
    "yellow_cards",
    "red_cards",

    # Balón largo
    "long_balls",
    "long_balls_won",

    # Porteros
    "saves",
    "saves_inside_box",
    "goals_conceded",
]

PER90_COLS = []

for metric in PER90_SOURCE_COLS:

    total_col = f"total_{metric}"
    per90_col = f"{metric}_per90"

    player_tournament_features[per90_col] = (
        player_tournament_features[total_col]
        / player_tournament_features["total_valid_minutes"]
        * 90
    )

    PER90_COLS.append(per90_col)


# ------------------------------------------------------------
# 2. TASAS DE EFICIENCIA
# ------------------------------------------------------------

RATE_DEFINITIONS = {
    "pass_accuracy": (
        "total_accurate_passes",
        "total_passes",
    ),
    "cross_accuracy": (
        "total_accurate_crosses",
        "total_total_crosses",
    ),
    "dribble_success": (
        "total_successful_dribbles",
        "total_dribble_attempts",
    ),
    "duel_success": (
        "total_duels_won",
        "total_total_duels",
    ),
    "aerial_success": (
        "total_aerials_won",
        "total_aerials",
    ),
    "long_ball_success": (
        "total_long_balls_won",
        "total_long_balls",
    ),
}

for rate, (numerator, denominator) in RATE_DEFINITIONS.items():

    player_tournament_features[rate] = (
        player_tournament_features[numerator]
        / player_tournament_features[denominator].replace(
            0,
            np.nan,
        )
    )


# ------------------------------------------------------------
# 3. SAVE RATE
# ------------------------------------------------------------

player_tournament_features["shots_faced_on_target"] = (
    player_tournament_features["total_saves"]
    + player_tournament_features["total_goals_conceded"]
)

player_tournament_features["save_rate"] = (
    player_tournament_features["total_saves"]
    / player_tournament_features[
        "shots_faced_on_target"
    ].replace(0, np.nan)
)


# ------------------------------------------------------------
# 4. RATING SPORTMONKS PONDERADO POR MINUTOS
# ------------------------------------------------------------
# No forma parte del PPS ni del IGR.
# Se conserva únicamente para validación externa posterior.

provider_rating_aux = df_valid.copy()

provider_rating_aux["rating_x_minutes"] = (
    provider_rating_aux["provider_rating"]
    * provider_rating_aux["minutes"]
)

provider_rating_tournament = (
    provider_rating_aux
    .groupby(
        GROUP_COLS,
        as_index=False,
    )
    .agg(
        rating_x_minutes=(
            "rating_x_minutes",
            "sum",
        ),
        rating_minutes=(
            "minutes",
            lambda x: x[
                provider_rating_aux.loc[
                    x.index,
                    "provider_rating",
                ].notna()
            ].sum(),
        ),
    )
)

provider_rating_tournament[
    "provider_rating_tournament"
] = (
    provider_rating_tournament["rating_x_minutes"]
    / provider_rating_tournament[
        "rating_minutes"
    ].replace(0, np.nan)
)

player_tournament_features = (
    player_tournament_features
    .merge(
        provider_rating_tournament[
            GROUP_COLS
            + ["provider_rating_tournament"]
        ],
        on=GROUP_COLS,
        how="left",
        validate="one_to_one",
    )
)


# ------------------------------------------------------------
# 5. CONTROLES
# ------------------------------------------------------------

RATE_COLS = list(RATE_DEFINITIONS) + ["save_rate"]

rate_out_of_bounds = {
    col: int(
        (
            player_tournament_features[col].notna()
            & (
                (player_tournament_features[col] < 0)
                | (player_tournament_features[col] > 1)
            )
        ).sum()
    )
    for col in RATE_COLS
}

inf_per90 = int(
    np.isinf(
        player_tournament_features[PER90_COLS]
        .to_numpy(dtype=float)
    ).sum()
)

print("=== FEATURES DERIVADAS ===")
print("Registros:", len(player_tournament_features))
print("Métricas /90:", len(PER90_COLS))
print("Tasas:", len(RATE_COLS))

print("\n=== CONTROL /90 ===")
print("Minutos <= 0:",
      (player_tournament_features[
          "total_valid_minutes"
      ] <= 0).sum())
print("Valores infinitos:", inf_per90)

print("\n=== TASAS FUERA DE [0, 1] ===")
print(rate_out_of_bounds)

print("\n=== PROVIDER RATING ===")
print(
    player_tournament_features[
        "provider_rating_tournament"
    ].describe()
)

print(
    "\nProvider rating NULL:",
    player_tournament_features[
        "provider_rating_tournament"
    ].isna().sum(),
)

features_ok = (
    len(player_tournament_features) == 448
    and inf_per90 == 0
    and sum(rate_out_of_bounds.values()) == 0
    and player_tournament_features[
        "provider_rating_tournament"
    ].isna().sum() == 0
)

print(
    "\n¿Features derivadas correctas?:",
    features_ok,
)

assert features_ok, (
    "Las features derivadas no superaron "
    "los controles de reproducibilidad."
)

=== FEATURES DERIVADAS ===
Registros: 448
Métricas /90: 36
Tasas: 7

=== CONTROL /90 ===
Minutos <= 0: 0
Valores infinitos: 0

=== TASAS FUERA DE [0, 1] ===
{'pass_accuracy': 0, 'cross_accuracy': 0, 'dribble_success': 0, 'duel_success': 0, 'aerial_success': 0, 'long_ball_success': 0, 'save_rate': 0}

=== PROVIDER RATING ===
count    448.000000
mean       6.845110
std        0.469392
min        5.560000
25%        6.557500
50%        6.820000
75%        7.082348
max        9.290000
Name: provider_rating_tournament, dtype: float64

Provider rating NULL: 0

¿Features derivadas correctas?: True


In [8]:
# ============================================================
# PASO 8 — PRIORS DE RIVALES PARA SHRINKAGE
# ============================================================

PUMAS_TEAM_ID = 1
K_SHRINKAGE = 10


# ------------------------------------------------------------
# 1. COMPONENTES DE CADA TASA
# ------------------------------------------------------------

SHRINKAGE_RATE_DEFINITIONS = {
    "pass_accuracy": (
        "total_accurate_passes",
        "total_passes",
    ),
    "cross_accuracy": (
        "total_accurate_crosses",
        "total_total_crosses",
    ),
    "dribble_success": (
        "total_successful_dribbles",
        "total_dribble_attempts",
    ),
    "duel_success": (
        "total_duels_won",
        "total_total_duels",
    ),
    "aerial_success": (
        "total_aerials_won",
        "total_aerials",
    ),
    "long_ball_success": (
        "total_long_balls_won",
        "total_long_balls",
    ),
    "save_rate": (
        "total_saves",
        "shots_faced_on_target",
    ),
}


# ------------------------------------------------------------
# 2. TASAS APLICABLES POR POSICIÓN
# ------------------------------------------------------------

SHRINKAGE_RATES_BY_POSITION = {
    "GK": [
        "pass_accuracy",
        "long_ball_success",
        "save_rate",
    ],
    "DF": [
        "pass_accuracy",
        "cross_accuracy",
        "dribble_success",
        "duel_success",
        "aerial_success",
        "long_ball_success",
    ],
    "MF": [
        "pass_accuracy",
        "cross_accuracy",
        "dribble_success",
        "duel_success",
        "aerial_success",
        "long_ball_success",
    ],
    "FW": [
        "pass_accuracy",
        "cross_accuracy",
        "dribble_success",
        "duel_success",
        "aerial_success",
        "long_ball_success",
    ],
}


# ------------------------------------------------------------
# 3. CALCULAR PRIORS SOLO CON RIVALES
# ------------------------------------------------------------

rival_rows = player_tournament_features.loc[
    player_tournament_features["team_id"]
    != PUMAS_TEAM_ID
].copy()

prior_rows = []

for (
    season_name,
    tournament_name,
    position_group
), subset in rival_rows.groupby(
    [
        "season_name",
        "tournament_name",
        "position_group",
    ]
):

    applicable_rates = (
        SHRINKAGE_RATES_BY_POSITION.get(
            position_group,
            [],
        )
    )

    for rate in applicable_rates:

        success_col, opportunity_col = (
            SHRINKAGE_RATE_DEFINITIONS[rate]
        )

        valid = subset.loc[
            subset[success_col].notna()
            & subset[opportunity_col].notna()
            & (subset[opportunity_col] > 0)
        ]

        successes = valid[success_col].sum()
        opportunities = valid[opportunity_col].sum()

        prior = (
            successes / opportunities
            if opportunities > 0
            else np.nan
        )

        prior_rows.append(
            {
                "season_name": season_name,
                "tournament_name": tournament_name,
                "position_group": position_group,
                "rate": rate,
                "rival_successes": successes,
                "rival_opportunities": opportunities,
                "rival_prior": prior,
                "rival_players": len(valid),
            }
        )


rival_priors = pd.DataFrame(prior_rows)


# ------------------------------------------------------------
# 4. CONTROLES
# ------------------------------------------------------------

expected_prior_combinations = (
    sum(
        len(rates)
        for rates
        in SHRINKAGE_RATES_BY_POSITION.values()
    )
    * 2  # Apertura + Clausura
)

prior_duplicates = (
    rival_priors
    .duplicated(
        subset=[
            "season_name",
            "tournament_name",
            "position_group",
            "rate",
        ]
    )
    .sum()
)

invalid_priors = (
    rival_priors["rival_prior"].notna()
    & (
        (rival_priors["rival_prior"] < 0)
        | (rival_priors["rival_prior"] > 1)
    )
).sum()


print("=== PRIORS DE RIVALES ===")
print("K shrinkage:", K_SHRINKAGE)
print("Registros:", len(rival_priors))
print(
    "Combinaciones esperadas:",
    expected_prior_combinations,
)
print("Duplicados:", prior_duplicates)
print("Priors NULL:", rival_priors["rival_prior"].isna().sum())
print("Priors fuera de [0,1]:", invalid_priors)

print("\n=== COMBINACIONES POR POSICIÓN ===")

print(
    rival_priors
    .groupby("position_group")["rate"]
    .nunique()
    .sort_index()
)

print("\n=== PRIORS ===")

print(
    rival_priors[
        [
            "tournament_name",
            "position_group",
            "rate",
            "rival_players",
            "rival_opportunities",
            "rival_prior",
        ]
    ]
    .sort_values(
        [
            "tournament_name",
            "position_group",
            "rate",
        ]
    )
    .to_string(index=False)
)


priors_ok = (
    len(rival_priors)
    == expected_prior_combinations
    and prior_duplicates == 0
    and rival_priors["rival_prior"].isna().sum() == 0
    and invalid_priors == 0
)

print(
    "\n¿Priors construidos correctamente?:",
    priors_ok,
)

assert priors_ok, (
    "Los priors de rivales no superaron "
    "los controles."
)

=== PRIORS DE RIVALES ===
K shrinkage: 10
Registros: 42
Combinaciones esperadas: 42
Duplicados: 0
Priors NULL: 0
Priors fuera de [0,1]: 0

=== COMBINACIONES POR POSICIÓN ===
position_group
DF    6
FW    6
GK    3
MF    6
Name: rate, dtype: int64

=== PRIORS ===
tournament_name position_group              rate  rival_players  rival_opportunities  rival_prior
       Apertura             DF    aerial_success             62                154.0     0.603896
       Apertura             DF    cross_accuracy             37                104.0     0.288462
       Apertura             DF   dribble_success             35                 57.0     0.491228
       Apertura             DF      duel_success             73                474.0     0.594937
       Apertura             DF long_ball_success             70                366.0     0.483607
       Apertura             DF     pass_accuracy             73               3115.0     0.871589
       Apertura             FW    aerial_success    

In [9]:
# ============================================================
# PASO 9 — SHRINKAGE DE TASAS DE PUMAS
# ============================================================

# ------------------------------------------------------------
# 1. EXTRAER PERFILES DE PUMAS
# ------------------------------------------------------------

pumas_rows = (
    player_tournament_features.loc[
        player_tournament_features["team_id"]
        == PUMAS_TEAM_ID
    ]
    .copy()
)

print("=== PUMAS ANTES DEL SHRINKAGE ===")
print("Registros jugador × torneo:", len(pumas_rows))

print(
    pumas_rows
    .groupby(
        ["tournament_name", "position_group"]
    )
    .size()
)


# ------------------------------------------------------------
# 2. APLICAR SHRINKAGE
# ------------------------------------------------------------

pumas_tournament_rates = []

for _, row in pumas_rows.iterrows():

    position = row["position_group"]

    applicable_rates = (
        SHRINKAGE_RATES_BY_POSITION.get(
            position,
            [],
        )
    )

    for rate in applicable_rates:

        success_col, opportunity_col = (
            SHRINKAGE_RATE_DEFINITIONS[rate]
        )

        successes = row[success_col]
        opportunities = row[opportunity_col]

        prior_match = rival_priors.loc[
            (
                rival_priors["season_name"]
                == row["season_name"]
            )
            & (
                rival_priors["tournament_name"]
                == row["tournament_name"]
            )
            & (
                rival_priors["position_group"]
                == position
            )
            & (
                rival_priors["rate"]
                == rate
            )
        ]

        assert len(prior_match) == 1

        rival_prior = (
            prior_match["rival_prior"].iloc[0]
        )

        # --------------------------------------------
        # Tasa observada
        # --------------------------------------------

        observed_rate = (
            successes / opportunities
            if (
                pd.notna(successes)
                and pd.notna(opportunities)
                and opportunities > 0
            )
            else np.nan
        )

        # --------------------------------------------
        # Shrinkage
        # --------------------------------------------
        # Si no existen oportunidades observadas,
        # no fabricamos una tasa: permanece NA.

        shrunk_rate = (
            (
                successes
                + K_SHRINKAGE * rival_prior
            )
            / (
                opportunities
                + K_SHRINKAGE
            )
            if (
                pd.notna(successes)
                and pd.notna(opportunities)
                and opportunities > 0
            )
            else np.nan
        )

        pumas_tournament_rates.append(
            {
                "season_name":
                    row["season_name"],

                "tournament_name":
                    row["tournament_name"],

                "player_id":
                    row["player_id"],

                "player_name":
                    row["player_name"],

                "position_group":
                    position,

                "rate":
                    rate,

                "successes":
                    successes,

                "opportunities":
                    opportunities,

                "observed_rate":
                    observed_rate,

                "rival_prior":
                    rival_prior,

                "shrunk_rate":
                    shrunk_rate,
            }
        )


pumas_tournament_rates = pd.DataFrame(
    pumas_tournament_rates
)


# ------------------------------------------------------------
# 3. FORMATO ANCHO DE LAS TASAS AJUSTADAS
# ------------------------------------------------------------

pumas_shrunk_wide = (
    pumas_tournament_rates
    .pivot(
        index=[
            "season_name",
            "tournament_name",
            "player_id",
            "player_name",
            "position_group",
        ],
        columns="rate",
        values="shrunk_rate",
    )
    .reset_index()
)

pumas_shrunk_wide.columns.name = None


# Añadir sufijo explícito
rate_cols_to_rename = {
    rate: f"{rate}_shrunk"
    for rate
    in SHRINKAGE_RATE_DEFINITIONS
    if rate in pumas_shrunk_wide.columns
}

pumas_shrunk_wide = (
    pumas_shrunk_wide
    .rename(columns=rate_cols_to_rename)
)


# ------------------------------------------------------------
# 4. CONTROLES
# ------------------------------------------------------------

shrunk_cols = [
    col
    for col in pumas_shrunk_wide.columns
    if col.endswith("_shrunk")
]

invalid_shrunk = sum(
    (
        pumas_shrunk_wide[col].notna()
        & (
            (pumas_shrunk_wide[col] < 0)
            | (pumas_shrunk_wide[col] > 1)
        )
    ).sum()
    for col in shrunk_cols
)

duplicate_rates = (
    pumas_tournament_rates
    .duplicated(
        subset=[
            "season_name",
            "tournament_name",
            "player_id",
            "position_group",
            "rate",
        ]
    )
    .sum()
)


print("\n=== SHRINKAGE ===")
print(
    "Registros long:",
    len(pumas_tournament_rates),
)
print(
    "Perfiles Pumas wide:",
    len(pumas_shrunk_wide),
)
print(
    "Tasas shrunk disponibles:",
    len(shrunk_cols),
)
print(
    "Duplicados long:",
    duplicate_rates,
)
print(
    "Valores shrunk fuera de [0,1]:",
    invalid_shrunk,
)

print("\n=== TASAS SHRUNK ===")
print(shrunk_cols)

print("\n=== NULL POR TASA SHRUNK ===")
print(
    pumas_shrunk_wide[shrunk_cols]
    .isna()
    .sum()
    .sort_values(ascending=False)
)


# ------------------------------------------------------------
# 5. VALIDACIÓN FINAL
# ------------------------------------------------------------

shrinkage_ok = (
    len(pumas_rows) == 43
    and len(pumas_shrunk_wide) == 43
    and len(shrunk_cols) == 7
    and duplicate_rates == 0
    and invalid_shrunk == 0
)

print(
    "\n¿Shrinkage construido correctamente?:",
    shrinkage_ok,
)

assert shrinkage_ok, (
    "El shrinkage no superó "
    "los controles de reproducibilidad."
)

=== PUMAS ANTES DEL SHRINKAGE ===
Registros jugador × torneo: 43
tournament_name  position_group
Apertura         DF                8
                 FW                5
                 GK                1
                 MF                8
Clausura         DF                8
                 FW                7
                 GK                2
                 MF                4
dtype: int64

=== SHRINKAGE ===
Registros long: 249
Perfiles Pumas wide: 43
Tasas shrunk disponibles: 7
Duplicados long: 0
Valores shrunk fuera de [0,1]: 0

=== TASAS SHRUNK ===
['aerial_success_shrunk', 'cross_accuracy_shrunk', 'dribble_success_shrunk', 'duel_success_shrunk', 'long_ball_success_shrunk', 'pass_accuracy_shrunk', 'save_rate_shrunk']

=== NULL POR TASA SHRUNK ===
save_rate_shrunk            40
cross_accuracy_shrunk        9
aerial_success_shrunk        7
dribble_success_shrunk       7
long_ball_success_shrunk     4
duel_success_shrunk          3
pass_accuracy_shrunk         0
dtype: int

In [10]:
# ============================================================
# PASO 10 — POBLACIÓN DE REFERENCIA PARA PPS
# ============================================================

REFERENCE_MINUTES = 90


# ------------------------------------------------------------
# 1. FILTRAR POBLACIÓN DE REFERENCIA
# ------------------------------------------------------------

pps_reference = (
    player_tournament_features.loc[
        player_tournament_features["total_valid_minutes"]
        >= REFERENCE_MINUTES
    ]
    .copy()
)


# ------------------------------------------------------------
# 2. DISTRIBUCIÓN POR TORNEO Y POSICIÓN
# ------------------------------------------------------------

reference_summary = (
    pps_reference
    .groupby(
        ["tournament_name", "position_group"]
    )
    ["total_valid_minutes"]
    .agg(
        players="count",
        median_minutes="median",
        min_minutes="min",
        max_minutes="max",
    )
)


# ------------------------------------------------------------
# 3. PUMAS ELEGIBLES
# ------------------------------------------------------------

pumas_reference = (
    pps_reference.loc[
        pps_reference["team_id"] == PUMAS_TEAM_ID
    ]
    .copy()
)

pumas_reference_summary = (
    pumas_reference
    .groupby(
        ["tournament_name", "position_group"]
    )
    .size()
)


# ------------------------------------------------------------
# 4. IDENTIFICAR PUMAS EXCLUIDOS POR EXPOSICIÓN
# ------------------------------------------------------------

pumas_not_eligible = (
    pumas_rows.loc[
        pumas_rows["total_valid_minutes"]
        < REFERENCE_MINUTES,
        [
            "season_name",
            "tournament_name",
            "player_id",
            "player_name",
            "position_group",
            "appearances",
            "total_valid_minutes",
        ],
    ]
    .sort_values(
        [
            "tournament_name",
            "total_valid_minutes",
        ]
    )
)


# ------------------------------------------------------------
# 5. CONTROLES
# ------------------------------------------------------------

print("=== POBLACIÓN DE REFERENCIA PPS ===")
print("Umbral:", REFERENCE_MINUTES, "minutos")
print("Registros:", len(pps_reference))

print("\n=== DISTRIBUCIÓN ===")
print(reference_summary)

print("\n=== PUMAS ELEGIBLES ===")
print("Registros:", len(pumas_reference))
print(pumas_reference_summary)

print("\n=== PUMAS NO ELEGIBLES ===")
print(
    pumas_not_eligible.to_string(index=False)
)


# ------------------------------------------------------------
# 6. VALIDACIÓN
# ------------------------------------------------------------

expected_reference = {
    ("Apertura", "DF"): 69,
    ("Apertura", "FW"): 18,
    ("Apertura", "GK"): 17,
    ("Apertura", "MF"): 36,
    ("Clausura", "DF"): 60,
    ("Clausura", "FW"): 23,
    ("Clausura", "GK"): 18,
    ("Clausura", "MF"): 33,
}

counts = (
    pps_reference
    .groupby(
        ["tournament_name", "position_group"]
    )
    .size()
    .to_dict()
)

reference_ok = (
    len(pps_reference) == 274
    and len(pumas_reference) == 39
    and len(pumas_not_eligible) == 4
    and counts == expected_reference
)

print(
    "\n¿Población de referencia correcta?:",
    reference_ok,
)

assert reference_ok, (
    "La población de referencia no reproduce "
    "el estado esperado."
)

=== POBLACIÓN DE REFERENCIA PPS ===
Umbral: 90 minutos
Registros: 274

=== DISTRIBUCIÓN ===
                                players  median_minutes  min_minutes  \
tournament_name position_group                                         
Apertura        DF                   69            90.0           90   
                FW                   18            90.0           90   
                GK                   17            90.0           90   
                MF                   36            90.0           90   
Clausura        DF                   60            90.0           90   
                FW                   23            90.0           90   
                GK                   18            90.0           90   
                MF                   33            90.0           90   

                                max_minutes  
tournament_name position_group               
Apertura        DF                     1530  
                FW                     1292  
   

In [11]:
# ============================================================
# PASO 11 — PPS v1 (POSITIONAL PERFORMANCE SCORE)
# ============================================================

PPS_VERSION = "PPS_v1"


# ------------------------------------------------------------
# 1. MÉTRICAS POR POSICIÓN
# ------------------------------------------------------------

PPS_FEATURES = {

    "GK": [
        "saves_per90",
        "save_rate_shrunk",
        "passes_per90",
        "pass_accuracy_shrunk",
        "long_balls_per90",
        "long_ball_success_shrunk",
        "ball_recoveries_per90",
        "clearances_per90",
    ],

    "DF": [
        "tackles_per90",
        "interceptions_per90",
        "clearances_per90",
        "ball_recoveries_per90",
        "total_duels_per90",
        "duel_success_shrunk",
        "aerials_per90",
        "aerial_success_shrunk",
        "passes_per90",
        "pass_accuracy_shrunk",
        "passes_in_final_third_per90",
        "long_ball_success_shrunk",
        "successful_dribbles_per90",
        "dribble_success_shrunk",
        "key_passes_per90",
    ],

    "MF": [
        "ball_recoveries_per90",
        "tackles_per90",
        "interceptions_per90",
        "total_duels_per90",
        "duel_success_shrunk",
        "passes_per90",
        "pass_accuracy_shrunk",
        "passes_in_final_third_per90",
        "key_passes_per90",
        "assists_per90",
        "successful_dribbles_per90",
        "dribble_success_shrunk",
        "cross_accuracy_shrunk",
        "long_ball_success_shrunk",
        "shots_per90",
        "goals_per90",
    ],

    "FW": [
        "goals_per90",
        "assists_per90",
        "shots_per90",
        "shots_on_target_per90",
        "key_passes_per90",
        "successful_dribbles_per90",
        "dribble_success_shrunk",
        "total_duels_per90",
        "duel_success_shrunk",
        "aerials_per90",
        "aerial_success_shrunk",
        "passes_in_final_third_per90",
        "pass_accuracy_shrunk",
        "cross_accuracy_shrunk",
    ],
}


# ------------------------------------------------------------
# 2. PESOS POR POSICIÓN
# ------------------------------------------------------------

PPS_WEIGHTS = {

    "GK": {
        "saves_per90": 0.20,
        "save_rate_shrunk": 0.25,
        "passes_per90": 0.10,
        "pass_accuracy_shrunk": 0.15,
        "long_balls_per90": 0.05,
        "long_ball_success_shrunk": 0.10,
        "ball_recoveries_per90": 0.10,
        "clearances_per90": 0.05,
    },

    "DF": {
        "tackles_per90": 0.10,
        "interceptions_per90": 0.10,
        "clearances_per90": 0.10,
        "ball_recoveries_per90": 0.10,
        "total_duels_per90": 0.07,
        "duel_success_shrunk": 0.10,
        "aerials_per90": 0.05,
        "aerial_success_shrunk": 0.08,
        "passes_per90": 0.05,
        "pass_accuracy_shrunk": 0.08,
        "passes_in_final_third_per90": 0.05,
        "long_ball_success_shrunk": 0.05,
        "successful_dribbles_per90": 0.02,
        "dribble_success_shrunk": 0.02,
        "key_passes_per90": 0.03,
    },

    "MF": {
        "ball_recoveries_per90": 0.08,
        "tackles_per90": 0.06,
        "interceptions_per90": 0.05,
        "total_duels_per90": 0.06,
        "duel_success_shrunk": 0.08,
        "passes_per90": 0.08,
        "pass_accuracy_shrunk": 0.08,
        "passes_in_final_third_per90": 0.08,
        "key_passes_per90": 0.10,
        "assists_per90": 0.07,
        "successful_dribbles_per90": 0.06,
        "dribble_success_shrunk": 0.05,
        "cross_accuracy_shrunk": 0.04,
        "long_ball_success_shrunk": 0.04,
        "shots_per90": 0.03,
        "goals_per90": 0.04,
    },

    "FW": {
        "goals_per90": 0.18,
        "assists_per90": 0.10,
        "shots_per90": 0.10,
        "shots_on_target_per90": 0.10,
        "key_passes_per90": 0.08,
        "successful_dribbles_per90": 0.07,
        "dribble_success_shrunk": 0.06,
        "total_duels_per90": 0.05,
        "duel_success_shrunk": 0.06,
        "aerials_per90": 0.04,
        "aerial_success_shrunk": 0.04,
        "passes_in_final_third_per90": 0.05,
        "pass_accuracy_shrunk": 0.04,
        "cross_accuracy_shrunk": 0.03,
    },
}


# ------------------------------------------------------------
# 3. INCORPORAR SHRINKAGE AL BENCHMARK
# ------------------------------------------------------------
# Hasta ahora el shrinkage se calculó explícitamente para Pumas.
# Para construir PPS, necesitamos las tasas ajustadas para toda
# la población de referencia usando el prior rival correspondiente.

pps_scored = pps_reference.copy()

for rate in SHRINKAGE_RATE_DEFINITIONS:

    pps_scored[f"{rate}_shrunk"] = np.nan


for idx, row in pps_scored.iterrows():

    position = row["position_group"]

    for rate in SHRINKAGE_RATES_BY_POSITION.get(
        position,
        [],
    ):

        success_col, opportunity_col = (
            SHRINKAGE_RATE_DEFINITIONS[rate]
        )

        successes = row[success_col]
        opportunities = row[opportunity_col]

        prior_match = rival_priors.loc[
            (
                rival_priors["season_name"]
                == row["season_name"]
            )
            & (
                rival_priors["tournament_name"]
                == row["tournament_name"]
            )
            & (
                rival_priors["position_group"]
                == position
            )
            & (
                rival_priors["rate"] == rate
            )
        ]

        assert len(prior_match) == 1

        prior = prior_match["rival_prior"].iloc[0]

        if (
            pd.notna(successes)
            and pd.notna(opportunities)
            and opportunities > 0
        ):
            pps_scored.loc[
                idx,
                f"{rate}_shrunk",
            ] = (
                successes
                + K_SHRINKAGE * prior
            ) / (
                opportunities
                + K_SHRINKAGE
            )


# ------------------------------------------------------------
# 4. VALIDAR PESOS
# ------------------------------------------------------------

for position, weights in PPS_WEIGHTS.items():

    assert set(weights) == set(
        PPS_FEATURES[position]
    )

    assert np.isclose(
        sum(weights.values()),
        1.0,
    )


# ------------------------------------------------------------
# 5. CONVERTIR FEATURES A PERCENTILES
# ------------------------------------------------------------

for position, features in PPS_FEATURES.items():

    position_mask = (
        pps_scored["position_group"] == position
    )

    for tournament in [
        "Apertura",
        "Clausura",
    ]:

        group_mask = (
            position_mask
            & (
                pps_scored["tournament_name"]
                == tournament
            )
        )

        for feature in features:

            pct_col = f"{feature}__pct"

            pps_scored.loc[
                group_mask,
                pct_col,
            ] = (
                pps_scored.loc[
                    group_mask,
                    feature,
                ]
                .rank(
                    pct=True,
                    method="average",
                )
                * 100
            )


# ------------------------------------------------------------
# 6. CALCULAR PPS
# ------------------------------------------------------------
# Si una métrica es NA, no se sustituye por cero.
# El peso se redistribuye entre las métricas disponibles.

pps_scored["pps_v1"] = np.nan


for idx, row in pps_scored.iterrows():

    position = row["position_group"]
    weights = PPS_WEIGHTS[position]

    weighted_sum = 0.0
    available_weight = 0.0

    for feature, weight in weights.items():

        pct_col = f"{feature}__pct"
        value = row[pct_col]

        if pd.notna(value):

            weighted_sum += value * weight
            available_weight += weight

    if available_weight > 0:

        pps_scored.loc[
            idx,
            "pps_v1",
        ] = (
            weighted_sum
            / available_weight
        )


# ------------------------------------------------------------
# 7. RANK POSICIONAL
# ------------------------------------------------------------

pps_scored["pps_position_rank"] = (
    pps_scored
    .groupby(
        [
            "tournament_name",
            "position_group",
        ]
    )["pps_v1"]
    .rank(
        ascending=False,
        method="min",
    )
    .astype("Int64")
)


# ------------------------------------------------------------
# 8. CONTROLES
# ------------------------------------------------------------

weight_sums = {
    position: sum(weights.values())
    for position, weights
    in PPS_WEIGHTS.items()
}

pumas_pps = (
    pps_scored.loc[
        pps_scored["team_id"] == PUMAS_TEAM_ID
    ]
    .copy()
)

print("=== PPS v1 ===")
print("Versión:", PPS_VERSION)
print("Benchmark:", len(pps_scored))
print("Pumas evaluados:", len(pumas_pps))

print("\n=== PESOS ===")
print(weight_sums)

print("\n=== PPS PUMAS ===")
print(
    pumas_pps[
        [
            "tournament_name",
            "player_name",
            "position_group",
            "total_valid_minutes",
            "pps_v1",
            "pps_position_rank",
        ]
    ]
    .sort_values(
        [
            "tournament_name",
            "position_group",
            "pps_position_rank",
        ]
    )
    .to_string(
        index=False,
        formatters={
            "pps_v1": "{:.2f}".format,
        },
    )
)


# ------------------------------------------------------------
# 9. VALIDACIÓN
# ------------------------------------------------------------

pps_ok = (
    len(pps_scored) == 274
    and len(pumas_pps) == 39
    and pps_scored["pps_v1"].isna().sum() == 0
    and all(
        np.isclose(value, 1.0)
        for value in weight_sums.values()
    )
)

print(
    "\n¿PPS construido correctamente?:",
    pps_ok,
)

assert pps_ok, (
    "PPS v1 no superó los controles."
)

=== PPS v1 ===
Versión: PPS_v1
Benchmark: 274
Pumas evaluados: 39

=== PESOS ===
{'GK': 1.0, 'DF': 1.0, 'MF': 1.0, 'FW': 1.0}

=== PPS PUMAS ===
tournament_name                            player_name position_group  total_valid_minutes pps_v1  pps_position_rank
       Apertura                   Rubén Duarte Sánchez             DF                 1260  62.34                  9
       Apertura                  Pablo Bennevendo Peña             DF                 1477  52.51                 29
       Apertura              Ángel Amhed Azuaje García             DF                  700  49.19                 39
       Apertura             Nathanael Ananias da Silva             DF                 1530  46.81                 46
       Apertura             Pablo Alfonso Monroy Reyes             DF                  173  41.65                 57
       Apertura         Álvaro Anyiver Angulo Mosquera             DF                 1173  35.99                 64
       Apertura                 Rodr

In [12]:
# ============================================================
# PASO 12 — IGR v1 (ÍNDICE GLOBAL DE RENDIMIENTO)
# ============================================================

IGR_VERSION = "IGR_v1"
IGR_K_MINUTES = 450
IGR_NEUTRAL_POINT = 50


# ------------------------------------------------------------
# 1. PERCENTIL PPS DENTRO DE TORNEO × POSICIÓN
# ------------------------------------------------------------
# Este percentil crea un puente común entre posiciones.
#
# Ejemplo conceptual:
# benchmark_pps_pct = 80
# -> el PPS del jugador está aproximadamente en el percentil
#    80 frente a jugadores de su misma posición y torneo.

pps_scored["benchmark_pps_pct"] = (
    pps_scored
    .groupby(
        [
            "tournament_name",
            "position_group",
        ]
    )["pps_v1"]
    .rank(
        pct=True,
        method="average",
    )
    * 100
)


# ------------------------------------------------------------
# 2. EXTRAER PUMAS ELEGIBLES
# ------------------------------------------------------------

pumas_igr = (
    pps_scored.loc[
        pps_scored["team_id"] == PUMAS_TEAM_ID
    ]
    .copy()
)


# ------------------------------------------------------------
# 3. CONFIABILIDAD POR EXPOSICIÓN
# ------------------------------------------------------------
# Los minutos NO añaden puntos directamente.
#
# Determinan cuánto confiamos en el percentil observado.
#
# A mayor exposición:
# reliability -> 1
#
# A menor exposición:
# reliability -> 0
# y el índice se aproxima al punto neutral 50.

pumas_igr["igr_reliability"] = (
    pumas_igr["total_valid_minutes"]
    / (
        pumas_igr["total_valid_minutes"]
        + IGR_K_MINUTES
    )
)


# ------------------------------------------------------------
# 4. IGR v1
# ------------------------------------------------------------

pumas_igr["igr_v1"] = (
    IGR_NEUTRAL_POINT
    + (
        pumas_igr["benchmark_pps_pct"]
        - IGR_NEUTRAL_POINT
    )
    * pumas_igr["igr_reliability"]
)


# ------------------------------------------------------------
# 5. RANK GLOBAL DENTRO DEL TORNEO
# ------------------------------------------------------------

pumas_igr["igr_rank"] = (
    pumas_igr
    .groupby("tournament_name")["igr_v1"]
    .rank(
        ascending=False,
        method="min",
    )
    .astype("Int64")
)


# ------------------------------------------------------------
# 6. CONTROLES
# ------------------------------------------------------------

print("=== IGR v1 ===")

print("Versión:", IGR_VERSION)
print("K minutos:", IGR_K_MINUTES)
print("Punto neutral:", IGR_NEUTRAL_POINT)
print("Jugadores evaluados:", len(pumas_igr))

print("\n=== DISTRIBUCIÓN IGR ===")

print(
    pumas_igr["igr_v1"]
    .describe()
)

print("\n=== POR TORNEO ===")

print(
    pumas_igr
    .groupby("tournament_name")
    .agg(
        jugadores=("player_id", "count"),
        igr_min=("igr_v1", "min"),
        igr_median=("igr_v1", "median"),
        igr_max=("igr_v1", "max"),
    )
)


# ------------------------------------------------------------
# 7. TOP 10 IGR POR TORNEO
# ------------------------------------------------------------

igr_top10 = (
    pumas_igr.loc[
        pumas_igr["igr_rank"] <= 10,
        [
            "tournament_name",
            "igr_rank",
            "player_name",
            "position_group",
            "total_valid_minutes",
            "igr_v1",
            "pps_v1",
            "benchmark_pps_pct",
            "igr_reliability",
            "provider_rating_tournament",
        ],
    ]
    .sort_values(
        [
            "tournament_name",
            "igr_rank",
        ]
    )
)

print("\n=== TOP 10 IGR POR TORNEO ===")

print(
    igr_top10.to_string(
        index=False,
        formatters={
            "igr_v1": "{:.2f}".format,
            "pps_v1": "{:.2f}".format,
            "benchmark_pps_pct": "{:.2f}".format,
            "igr_reliability": "{:.3f}".format,
            "provider_rating_tournament":
                "{:.2f}".format,
        },
    )
)


# ------------------------------------------------------------
# 8. VALIDACIÓN
# ------------------------------------------------------------

igr_ok = (
    len(pumas_igr) == 39
    and pumas_igr["benchmark_pps_pct"].isna().sum() == 0
    and pumas_igr["igr_reliability"].isna().sum() == 0
    and pumas_igr["igr_v1"].isna().sum() == 0
    and pumas_igr["igr_rank"].isna().sum() == 0
    and len(igr_top10) == 20
    and (
        (pumas_igr["igr_reliability"] > 0)
        & (pumas_igr["igr_reliability"] < 1)
    ).all()
)

print(
    "\n¿IGR construido correctamente?:",
    igr_ok,
)

assert igr_ok, (
    "IGR v1 no superó los controles."
)

=== IGR v1 ===
Versión: IGR_v1
K minutos: 450
Punto neutral: 50
Jugadores evaluados: 39

=== DISTRIBUCIÓN IGR ===
count    39.000000
mean     53.988873
std      15.958147
min      20.147874
25%      41.241936
50%      56.085132
75%      65.476599
max      81.756568
Name: igr_v1, dtype: float64

=== POR TORNEO ===
                 jugadores    igr_min  igr_median    igr_max
tournament_name                                             
Apertura                19  20.147874   56.975462  78.299008
Clausura                20  28.518519   55.729672  81.756568

=== TOP 10 IGR POR TORNEO ===
tournament_name  igr_rank                            player_name position_group  total_valid_minutes igr_v1 pps_v1 benchmark_pps_pct igr_reliability provider_rating_tournament
       Apertura         1                   Rubén Duarte Sánchez             DF                 1260  78.30  62.34             88.41           0.737                       6.97
       Apertura         2                    Keylor Navas 

In [13]:
# ============================================================
# PASO 13 — VALIDACIÓN EXTERNA CON PROVIDER RATING
# ============================================================

VALIDATION_COLS = [
    "pps_v1",
    "benchmark_pps_pct",
    "igr_v1",
    "provider_rating_tournament",
]


# ------------------------------------------------------------
# 1. CONTROL DE DATOS
# ------------------------------------------------------------

validation_data = (
    pumas_igr[
        [
            "season_name",
            "tournament_name",
            "player_id",
            "player_name",
            "position_group",
            "total_valid_minutes",
        ]
        + VALIDATION_COLS
    ]
    .dropna(subset=VALIDATION_COLS)
    .copy()
)


print("=== VALIDACIÓN EXTERNA ===")
print("Registros disponibles:", len(validation_data))
print(
    "Provider rating NULL:",
    pumas_igr["provider_rating_tournament"]
    .isna()
    .sum(),
)


# ------------------------------------------------------------
# 2. FUNCIÓN DE CORRELACIÓN
# ------------------------------------------------------------

def correlation_summary(data, metric):
    return {
        "metric": metric,
        "n": len(data),

        "pearson": data[
            [metric, "provider_rating_tournament"]
        ].corr(method="pearson").iloc[0, 1],

        "spearman": data[
            [metric, "provider_rating_tournament"]
        ].corr(method="spearman").iloc[0, 1],
    }


# ------------------------------------------------------------
# 3. VALIDACIÓN GLOBAL PUMAS
# ------------------------------------------------------------

global_validation = pd.DataFrame(
    [
        correlation_summary(
            validation_data,
            metric,
        )
        for metric in [
            "pps_v1",
            "benchmark_pps_pct",
            "igr_v1",
        ]
    ]
)


print("\n=== GLOBAL PUMAS ===")

print(
    global_validation.to_string(
        index=False,
        formatters={
            "pearson": "{:.3f}".format,
            "spearman": "{:.3f}".format,
        },
    )
)


# ------------------------------------------------------------
# 4. VALIDACIÓN POR TORNEO
# ------------------------------------------------------------

tournament_validation_rows = []

for tournament, subset in (
    validation_data
    .groupby("tournament_name")
):

    for metric in [
        "pps_v1",
        "benchmark_pps_pct",
        "igr_v1",
    ]:

        result = correlation_summary(
            subset,
            metric,
        )

        result["tournament_name"] = tournament

        tournament_validation_rows.append(
            result
        )


tournament_validation = pd.DataFrame(
    tournament_validation_rows
)


print("\n=== POR TORNEO ===")

print(
    tournament_validation[
        [
            "tournament_name",
            "metric",
            "n",
            "pearson",
            "spearman",
        ]
    ]
    .to_string(
        index=False,
        formatters={
            "pearson": "{:.3f}".format,
            "spearman": "{:.3f}".format,
        },
    )
)


# ------------------------------------------------------------
# 5. VALIDACIÓN PPS EN TODO EL BENCHMARK
# ------------------------------------------------------------
# Aquí podemos usar los 274 perfiles, no sólo Pumas.
# Sirve para evaluar la relación general entre PPS y
# provider rating dentro de nuestra población de referencia.

benchmark_validation = (
    pps_scored[
        [
            "pps_v1",
            "provider_rating_tournament",
        ]
    ]
    .dropna()
)

benchmark_pearson = (
    benchmark_validation[
        [
            "pps_v1",
            "provider_rating_tournament",
        ]
    ]
    .corr(method="pearson")
    .iloc[0, 1]
)

benchmark_spearman = (
    benchmark_validation[
        [
            "pps_v1",
            "provider_rating_tournament",
        ]
    ]
    .corr(method="spearman")
    .iloc[0, 1]
)


print("\n=== PPS — BENCHMARK COMPLETO ===")

print(
    "N:",
    len(benchmark_validation),
)

print(
    "Pearson:",
    round(benchmark_pearson, 3),
)

print(
    "Spearman:",
    round(benchmark_spearman, 3),
)


# ------------------------------------------------------------
# 6. CONTROLES
# ------------------------------------------------------------

validation_ok = (
    len(validation_data) == 39
    and len(benchmark_validation) == 274
    and global_validation[
        ["pearson", "spearman"]
    ].notna().all().all()
    and tournament_validation[
        ["pearson", "spearman"]
    ].notna().all().all()
    and pd.notna(benchmark_pearson)
    and pd.notna(benchmark_spearman)
)


print(
    "\n¿Validación calculada correctamente?:",
    validation_ok,
)

assert validation_ok, (
    "La validación externa no superó "
    "los controles."
)


=== VALIDACIÓN EXTERNA ===
Registros disponibles: 39
Provider rating NULL: 0

=== GLOBAL PUMAS ===
           metric  n pearson spearman
           pps_v1 39   0.603    0.574
benchmark_pps_pct 39   0.616    0.606
           igr_v1 39   0.576    0.575

=== POR TORNEO ===
tournament_name            metric  n pearson spearman
       Apertura            pps_v1 19   0.527    0.544
       Apertura benchmark_pps_pct 19   0.574    0.534
       Apertura            igr_v1 19   0.429    0.475
       Clausura            pps_v1 20   0.725    0.668
       Clausura benchmark_pps_pct 20   0.706    0.675
       Clausura            igr_v1 20   0.688    0.716

=== PPS — BENCHMARK COMPLETO ===
N: 274
Pearson: 0.59
Spearman: 0.592

¿Validación calculada correctamente?: True


In [14]:
# ============================================================
# PASO 14 — TABLA MAESTRA FINAL DE EVALUACIÓN
# ============================================================

FINAL_COLUMNS = [
    "season_name",
    "tournament_name",
    "player_id",
    "player_name",
    "position_group",
    "appearances",
    "total_valid_minutes",
    "pps_v1",
    "pps_position_rank",
    "benchmark_pps_pct",
    "igr_reliability",
    "igr_v1",
    "igr_rank",
    "provider_rating_tournament",
]


# ------------------------------------------------------------
# 1. CONSTRUIR TABLA FINAL
# ------------------------------------------------------------

pumas_evaluation_final = (
    pumas_igr[FINAL_COLUMNS]
    .copy()
    .sort_values(
        [
            "tournament_name",
            "igr_rank",
        ]
    )
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# 2. TOP 10 IGR PARA PRESENTACIÓN
# ------------------------------------------------------------

igr_top10_final = (
    pumas_evaluation_final.loc[
        pumas_evaluation_final["igr_rank"] <= 10,
        [
            "tournament_name",
            "igr_rank",
            "player_name",
            "position_group",
            "total_valid_minutes",
            "igr_v1",
            "pps_v1",
            "provider_rating_tournament",
        ],
    ]
    .copy()
    .rename(
        columns={
            "tournament_name": "Torneo",
            "igr_rank": "Rank",
            "player_name": "Jugador",
            "position_group": "Posición",
            "total_valid_minutes": "Minutos",
            "igr_v1": "IGR",
            "pps_v1": "PPS",
            "provider_rating_tournament":
                "Rating proveedor",
        }
    )
)


# ------------------------------------------------------------
# 3. CONTROLES DE INTEGRIDAD
# ------------------------------------------------------------

duplicate_final = (
    pumas_evaluation_final
    .duplicated(
        subset=[
            "season_name",
            "tournament_name",
            "player_id",
        ]
    )
    .sum()
)

null_pps = (
    pumas_evaluation_final["pps_v1"]
    .isna()
    .sum()
)

null_igr = (
    pumas_evaluation_final["igr_v1"]
    .isna()
    .sum()
)

null_provider = (
    pumas_evaluation_final[
        "provider_rating_tournament"
    ]
    .isna()
    .sum()
)

tournament_counts = (
    pumas_evaluation_final
    .groupby("tournament_name")
    .size()
    .to_dict()
)

top10_counts = (
    igr_top10_final
    .groupby("Torneo")
    .size()
    .to_dict()
)


# ------------------------------------------------------------
# 4. RESUMEN
# ------------------------------------------------------------

print("=== TABLA MAESTRA FINAL ===")
print("Registros:", len(pumas_evaluation_final))
print("Columnas:", len(pumas_evaluation_final.columns))
print("Duplicados:", duplicate_final)
print("PPS NULL:", null_pps)
print("IGR NULL:", null_igr)
print("Provider rating NULL:", null_provider)

print("\n=== REGISTROS POR TORNEO ===")
print(tournament_counts)

print("\n=== TOP 10 POR TORNEO ===")
print(top10_counts)


# ------------------------------------------------------------
# 5. TABLA DE PRESENTACIÓN
# ------------------------------------------------------------

print("\n=== TOP 10 IGR — RESULTADO FINAL ===")

print(
    igr_top10_final.to_string(
        index=False,
        formatters={
            "IGR": "{:.2f}".format,
            "PPS": "{:.2f}".format,
            "Rating proveedor": "{:.2f}".format,
        },
    )
)


# ------------------------------------------------------------
# 6. VALIDACIÓN FINAL
# ------------------------------------------------------------

final_ok = (
    len(pumas_evaluation_final) == 39
    and tournament_counts
        == {
            "Apertura": 19,
            "Clausura": 20,
        }
    and duplicate_final == 0
    and null_pps == 0
    and null_igr == 0
    and null_provider == 0
    and len(igr_top10_final) == 20
    and top10_counts
        == {
            "Apertura": 10,
            "Clausura": 10,
        }
)

print(
    "\n¿Tabla maestra final correcta?:",
    final_ok,
)

assert final_ok, (
    "La tabla maestra final no superó "
    "los controles."
)

=== TABLA MAESTRA FINAL ===
Registros: 39
Columnas: 14
Duplicados: 0
PPS NULL: 0
IGR NULL: 0
Provider rating NULL: 0

=== REGISTROS POR TORNEO ===
{'Apertura': 19, 'Clausura': 20}

=== TOP 10 POR TORNEO ===
{'Apertura': 10, 'Clausura': 10}

=== TOP 10 IGR — RESULTADO FINAL ===
  Torneo  Rank                                Jugador Posición  Minutos   IGR   PPS Rating proveedor
Apertura     1                   Rubén Duarte Sánchez       DF     1260 78.30 62.34             6.97
Apertura     2                    Keylor Navas Gamboa       GK     1350 74.26 70.79             6.83
Apertura     3              José Luis Caicedo Barrera       MF     1154 67.99 58.43             6.68
Apertura     4                        Jorge Ruvalcaba       FW     1292 66.48 56.33             6.99
Apertura     5                José Juan Macías Guzmán       FW      218 64.50 70.64             7.04
Apertura     6           Piero Aldair  Quispe Córdova       MF      127 61.01 67.86             6.83
Apertura     7 

In [15]:
# ============================================================
# PASO 15 — EXPORTACIÓN DE RESULTADOS FINALES
# ============================================================

OUTPUT_DIR = Path("../data/processed")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


# ------------------------------------------------------------
# 1. RUTAS DE SALIDA
# ------------------------------------------------------------

MASTER_PATH = (
    OUTPUT_DIR
    / "pumas_player_evaluation_2025_2026.csv"
)

TOP10_PATH = (
    OUTPUT_DIR
    / "pumas_igr_top10_2025_2026.csv"
)

VALIDATION_PATH = (
    OUTPUT_DIR
    / "pumas_model_validation_2025_2026.csv"
)


# ------------------------------------------------------------
# 2. EXPORTAR TABLA MAESTRA
# ------------------------------------------------------------

pumas_evaluation_final.to_csv(
    MASTER_PATH,
    index=False,
    encoding="utf-8",
)


# ------------------------------------------------------------
# 3. EXPORTAR TOP 10 IGR
# ------------------------------------------------------------

igr_top10_final.to_csv(
    TOP10_PATH,
    index=False,
    encoding="utf-8",
)


# ------------------------------------------------------------
# 4. EXPORTAR VALIDACIÓN
# ------------------------------------------------------------

validation_export = (
    tournament_validation[
        [
            "tournament_name",
            "metric",
            "n",
            "pearson",
            "spearman",
        ]
    ]
    .copy()
)

global_export = (
    global_validation.copy()
)

global_export.insert(
    0,
    "tournament_name",
    "GLOBAL_PUMAS",
)

validation_export = pd.concat(
    [
        global_export[
            [
                "tournament_name",
                "metric",
                "n",
                "pearson",
                "spearman",
            ]
        ],
        validation_export,
    ],
    ignore_index=True,
)

validation_export.to_csv(
    VALIDATION_PATH,
    index=False,
    encoding="utf-8",
)


# ------------------------------------------------------------
# 5. COMPROBAR ARCHIVOS
# ------------------------------------------------------------

export_checks = {
    "master_exists":
        MASTER_PATH.exists(),

    "top10_exists":
        TOP10_PATH.exists(),

    "validation_exists":
        VALIDATION_PATH.exists(),

    "master_rows":
        len(pd.read_csv(MASTER_PATH)),

    "top10_rows":
        len(pd.read_csv(TOP10_PATH)),

    "validation_rows":
        len(pd.read_csv(VALIDATION_PATH)),
}


print("=== EXPORTACIÓN FINAL ===")

for key, value in export_checks.items():
    print(f"{key}: {value}")


print("\n=== ARCHIVOS ===")
print(MASTER_PATH)
print(TOP10_PATH)
print(VALIDATION_PATH)


# ------------------------------------------------------------
# 6. VALIDACIÓN
# ------------------------------------------------------------

export_ok = (
    export_checks["master_exists"]
    and export_checks["top10_exists"]
    and export_checks["validation_exists"]
    and export_checks["master_rows"] == 39
    and export_checks["top10_rows"] == 20
    and export_checks["validation_rows"] == 9
)

print(
    "\n¿Exportación correcta?:",
    export_ok,
)

assert export_ok, (
    "La exportación final no superó los controles."
)

=== EXPORTACIÓN FINAL ===
master_exists: True
top10_exists: True
validation_exists: True
master_rows: 39
top10_rows: 20
validation_rows: 9

=== ARCHIVOS ===
../data/processed/pumas_player_evaluation_2025_2026.csv
../data/processed/pumas_igr_top10_2025_2026.csv
../data/processed/pumas_model_validation_2025_2026.csv

¿Exportación correcta?: True
